In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision  # Torchvision is a library in PyTorch used for computer vision tasks
                    # a) It provides popular datasets like CIFAR-10, MNIST(dataset of handwritten digits (0–9)), ImageNet, etc.
                    # b) It also provides pretrained CNN models (like ResNet, VGG, AlexNet) , these models are already trained on millions 
                    #    of images, which is difficult to achieve using normal CPUs or basic GPUs
                    # c) It includes utilities (tools) for image processing such as:
                    #    - Transformations (resize, normalize, augment images)
                          # e.g., images with pixel values in range 0–255 are first scaled to 0–1, and then normalized to a range 
                          # like −1 to +1 before feeding them into a CNN to improve model training and performance.
                    #    - Loading and saving images
                    #    - Converting images to tensors

                    #  Overall, torchvision makes it easier to work with image data in deep learning
from torchvision.datasets import CIFAR10

In [2]:
# What is the CIFAR-10 dataset?
# CIFAR-10 is a very popular dataset used to train and test image classification models (like CNNs)

# Key Details
# Total images: 60,000
# Image size: 32 × 32 × 3 (RGB)
# Training set: 50,000 images
# Test set: 10,000 images
# Classes: 10 categories (Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck )

### DataSets and DataLoaders

In [3]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms # Transforms are used to preprocess images before feeding them into a model

# image => scale(0,1) => normalize => (-1,1)
transform = transforms.Compose([ # combines multiple preprocessing steps and applies them sequentially to an image 
                                 # Basically instead of applying transforms one by one ,we do it in one step using Compose
    transforms.ToTensor(), # automatically converts images into pytorch tensors and also automatically scale the images to 0-1
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # transforms.Normalize((mean values), (std values))
])

# When we need training dataset(Train = True),and for testing dataset(Train =False)
trainset = CIFAR10(root="./data", train=True, download=True, transform=transform) 
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [4]:
# How we do normalization in images?
# We do by transforms.Normalize((mean values), (std values)) ,it actually normalizes each channel (R, G, B) of the image
# Formula used: output = (input - mean) / std
# Above we set mean = (0.5, 0.5, 0.5) and std  = (0.5, 0.5, 0.5) ,this applied separately to each channel (R, G, B)
# Steps:
# a) Conversion of images into pytorch tensors by .ToTensor() and also automatically scale the images, pixel values from [0, 255] → [0, 1]
# b) Apply normalization: (value - mean) / std

# For Example
# value = 0   → (0 - 0.5)/0.5 = -1  
# value = 0.5 → (0.5 - 0.5)/0.5 = 0  
# value = 1   → (1 - 0.5)/0.5 = +1  

# Output in range --> [-1,1]

# It normalizes image pixel values from [0,1] to [-1,1] by subtracting 0.5 and dividing by 0.5 for each channel

In [5]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
testset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [7]:
# why we are not using TensorDataset here?
# We don’t use TensorDataset here because CIFAR10 is already a Dataset , Images are converted to tensors using transforms
# DataLoader directly works with this dataset
# TensorDataset is only needed when your input data and labels are already PyTorch tensors stored in memory
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Built the CNN 

In [8]:
# We know the input image size is 32×32×3 (H×W×C). We rearrange it to 3×32×32 (C×H×W) before passing it to the CNN
# Because Original image format → Height × Width × Channels (HWC) and PyTorch CNN expects → Channels × Height × Width (CHW)
# This rearrangement is automatically done by: transforms.ToTensor()

In [9]:
# For 1st Block(Input Image size(n) = 3x32x32, padding(p) = 1, kernel size(f) = 3 , stride(s) = 1)
# Output size = (n-f+2p)/s +1 = (32-3+2)/1 +1 = 32

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # This creates a pipeline of layers that will process an image step by step
        # Input → Conv → ReLU → Pool → Conv → ReLU → Pool → Conv → ReLU → Pool
        self.layers = nn.Sequential(
            # 1st Block
            # We use nn.Conv2d (2D convolution layer) because images have two spatial dimensions (height and width), 
            # and the convolution operation is applied over these 2D dimensions
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Kernel size(filter)= 3x3
            # nn.conv2d(here 3 is numper of input channels ,for RGB image → 3 and Grayscale → 1)
            # nn.conv2d(here 32 means number of filters we are applying or how many features maps we get)
            # Feature Maps -> a feature map is the ouput we get after applying a filter(kernel) to an image
            nn.ReLU(),
           # After applying the convolution layer and ReLU, the image size changes from (3 × 32 × 32) to (32 × 32 × 32),which becomes the input 
           # to the pooling layer because the convolution uses 32 filters, each producing one feature map,resulting in 32 feature maps (channels)
            nn.MaxPool2d(2, 2), # kernel size = 2x2, stride = 2

            # 2nd Block
            # you will see we double the feature maps as we go deep because
            # Early layers: Large image, simple features → fewer filters needed
            # Deeper layers: Small image, complex features → more filters needed
            # So, We increase (often double) the number of feature maps in deeper layers to capture more complex features while the spatial size decreases
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10) # Since this is a multi-class classification problem with 10 classes, the output layer should have 10 neurons
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [11]:
model = CNN()

In [12]:
# In multi-class classification using CNN, we have: Input → Conv Layers → ReLU → Pooling → Flatten → Fully Connected → Output Layer
# In hidden layers, CNNs use convolutional layers and fully connected (linear) layers, typically combined with activation functions like ReLU.
# We have learned that Softmax is applied in the output layer to get class probabilities.

# However, in PyTorch, when we use CrossEntropyLoss as the loss function, it internally applies Softmax to the model outputs and then computes the loss.
# Therefore, we should NOT apply Softmax manually in the output layer, because it is already handled by CrossEntropyLoss.

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training the CNN

In [13]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3639334356388473
epoch=2/10 & loss=0.9233360275283189
epoch=3/10 & loss=0.7298718054809838
epoch=4/10 & loss=0.602795805467669
epoch=5/10 & loss=0.49961155430054116
epoch=6/10 & loss=0.39789939427848364
epoch=7/10 & loss=0.3097828978772663
epoch=8/10 & loss=0.23820114945111526
epoch=9/10 & loss=0.18357179468483342
epoch=10/10 & loss=0.14658231979421796


## Evaluating our CNN

In [14]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.5
